In [4]:
import os
import json
import boto3
import psycopg2
import os
import requests
from datetime import datetime, timedelta
print("importaciones con éxito")

importaciones con éxito


In [2]:
# CELDA 8: CREAR SCRIPT SQL PARA RDS EN AWS
# Fuente: Adaptado de 06.CREATE_INSERT.ipynb + 04.Constraints.ipynb

rds_sql_script = '''
-- SCRIPT SQL PARA RDS EN AWS: BASE DE DATOS RAWG - VIDEOJUEGOS
-- Autor: Manuel Serrano (rama manel-rawg)
-- Objetivo: Esquema mínimo funcional para RDS en AWS
-- Nota: Este script se ejecutará en la instancia RDS, no en localhost

-- PASO 1: CREAR TABLAS MAESTRAS
CREATE TABLE IF NOT EXISTS esrb_ratings (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS genres (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS platforms (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

-- PASO 2: CREAR TABLA PRINCIPAL (games)
CREATE TABLE IF NOT EXISTS games (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    released DATE,
    rating NUMERIC(3,2),
    ratings_count INTEGER,
    metacritic INTEGER,
    playtime INTEGER,
    status_yet INTEGER DEFAULT 0,
    status_owned INTEGER DEFAULT 0,
    status_beaten INTEGER DEFAULT 0,
    status_toplay INTEGER DEFAULT 0,
    status_dropped INTEGER DEFAULT 0,
    status_playing INTEGER DEFAULT 0,
    success BOOLEAN DEFAULT FALSE,
    esrb_rating_id INTEGER REFERENCES esrb_ratings(id) ON DELETE SET NULL,
    updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Índices para optimización
CREATE INDEX IF NOT EXISTS idx_games_rating ON games(rating DESC);
CREATE INDEX IF NOT EXISTS idx_games_success ON games(success);
CREATE INDEX IF NOT EXISTS idx_games_esrb ON games(esrb_rating_id);

-- PASO 3: CREAR TABLAS DE RELACIÓN
CREATE TABLE IF NOT EXISTS game_genres (
    game_id INTEGER NOT NULL REFERENCES games(id) ON DELETE CASCADE,
    genre_id INTEGER NOT NULL REFERENCES genres(id) ON DELETE CASCADE,
    PRIMARY KEY (game_id, genre_id)
);

CREATE TABLE IF NOT EXISTS game_platforms (
    game_id INTEGER NOT NULL REFERENCES games(id) ON DELETE CASCADE,
    platform_id INTEGER NOT NULL REFERENCES platforms(id) ON DELETE CASCADE,
    released_at DATE,
    PRIMARY KEY (game_id, platform_id)
);

-- PASO 4: VISTA Y FUNCIÓN
CREATE OR REPLACE VIEW games_for_ml AS
SELECT 
    g.id, g.name, EXTRACT(YEAR FROM g.released) AS release_year,
    g.rating, g.ratings_count, g.metacritic, g.playtime,
    g.status_yet, g.status_owned, g.status_beaten, g.status_toplay, g.status_dropped, g.status_playing,
    g.success, er.name AS esrb_rating,
    STRING_AGG(DISTINCT ge.name, ', ') AS genres_list,
    STRING_AGG(DISTINCT p.name, ', ') AS platforms_list
FROM games g
LEFT JOIN esrb_ratings er ON g.esrb_rating_id = er.id
LEFT JOIN game_genres gg ON g.id = gg.game_id
LEFT JOIN genres ge ON gg.genre_id = ge.id
LEFT JOIN game_platforms gp ON g.id = gp.game_id
LEFT JOIN platforms p ON gp.platform_id = p.id
GROUP BY g.id, g.name, g.released, g.rating, g.ratings_count, g.metacritic, 
         g.playtime, g.status_yet, g.status_owned, g.status_beaten, 
         g.status_toplay, g.status_dropped, g.status_playing, g.success, er.name;

CREATE OR REPLACE FUNCTION calculate_success() 
RETURNS VOID AS $$
BEGIN
    UPDATE games 
    SET success = CASE 
        WHEN rating >= 4.0 AND ratings_count >= 1000 THEN TRUE
        ELSE FALSE
    END
    WHERE rating IS NOT NULL AND ratings_count IS NOT NULL;
END;
$$ LANGUAGE plpgsql;

-- Datos iniciales para ESRB
INSERT INTO esrb_ratings (id, name) VALUES
    (1, 'Everyone'), (2, 'Everyone 10+'), (3, 'Teen'),
    (4, 'Mature'), (5, 'Adults Only'), (6, 'Rating Pending')
ON CONFLICT (id) DO NOTHING;

-- FIN DEL SCRIPT PARA RDS
'''

# Guardar script para RDS
with open("02_create_rawg_database_rds.sql", "w", encoding="utf-8") as f:
    f.write(rds_sql_script)

print("✅ Celda 8 ejecutada correctamente")
print("   - Script SQL para RDS generado: 02_create_rawg_database_rds.sql")
print("   - Listo para ejecutar en instancia RDS en AWS")

✅ Celda 8 ejecutada correctamente
   - Script SQL para RDS generado: 02_create_rawg_database_rds.sql
   - Listo para ejecutar en instancia RDS en AWS


In [5]:
# CELDA 9: CREAR ESTRUCTURA DE CARPETAS PARA LAMBDAS
# Fuente: Adaptado de 20.AWS - LAMBDA.ipynb

# Crear carpetas
lambda_dirs = [
    "src/lambda_daily",
    "src/lambda_load_to_rds"
]

for dir_path in lambda_dirs:
    os.makedirs(dir_path, exist_ok=True)
    print(f"✅ Carpeta creada: {dir_path}")

# Crear requirements.txt
requirements_content = """psycopg2-binary==2.9.7
requests==2.31.0
boto3==1.28.62
pandas==2.1.1
SQLAlchemy==2.0.23"""

for dir_path in lambda_dirs:
    with open(f"{dir_path}/requirements.txt", "w") as f:
        f.write(requirements_content)
    print(f"✅ requirements.txt creado: {dir_path}/requirements.txt")

print("\n✅ Celda 9 ejecutada correctamente")
print("   - Estructura de carpetas para Lambdas creada")
print("   - requirements.txt generados para todas las Lambdas")

✅ Carpeta creada: src/lambda_daily
✅ Carpeta creada: src/lambda_load_to_rds
✅ requirements.txt creado: src/lambda_daily/requirements.txt
✅ requirements.txt creado: src/lambda_load_to_rds/requirements.txt

✅ Celda 9 ejecutada correctamente
   - Estructura de carpetas para Lambdas creada
   - requirements.txt generados para todas las Lambdas


In [13]:
# CELDA 10: CREAR HANDLER PARA LAMBDA 3 (CARGA A RDS)
# Fuente: Adaptado de 20.AWS - LAMBDA.ipynb

lambda3_handler = '''

def safe_int(value, default=0, max_val=2147483647):
    try:
        if value is None:
            return default
        val = int(float(value))
        return min(max(val, 0), max_val)
    except (ValueError, TypeError):
        return default

def safe_numeric(value, default=0.0, max_val=5.0):
    try:
        if value is None:
            return default
        val = float(value)
        return min(max(val, 0.0), max_val)
    except (ValueError, TypeError):
        return default

def lambda_handler(event, context):
    """Lambda 3: Carga datos desde S3 a RDS"""
    try:
        # 1. Obtener información del evento S3
        bucket = event["Records"][0]["s3"]["bucket"]["name"]
        key = event["Records"][0]["s3"]["object"]["key"]
        
        print(f"Procesando archivo: s3://{bucket}/{key}")  # Sin emoji para evitar problemas
        
        # 2. Descargar archivo JSON de S3
        s3_client = boto3.client("s3")
        response = s3_client.get_object(Bucket=bucket, Key=key)
        data = json.loads(response["Body"].read())
        
        # 3. Conectar a RDS
        conn = psycopg2.connect(
            host=os.environ["DB_HOST"],
            dbname=os.environ["DB_NAME"],
            user=os.environ["DB_USER"],
            password=os.environ["DB_PASSWORD"],
            port=os.environ.get("DB_PORT", "5432")
        )
        cursor = conn.cursor()
        
        # 4. Insertar datos en RDS
        games_data = []
        for game in data:  # ✅ CORRECCIÓN CRÍTICA: 'for game in data'
            added_status = game.get("added_by_status") or {}
            esrb_rating = game.get("esrb_rating") or {}
            
            games_data.append((
                safe_int(game["id"]),
                str(game["name"]) if game["name"] else "Unknown",
                game.get("released"),
                safe_numeric(game.get("rating")),
                safe_int(game.get("ratings_count")),
                safe_int(game.get("metacritic"), max_val=100),
                safe_int(game.get("playtime")),
                safe_int(added_status.get("yet", 0)),
                safe_int(added_status.get("owned", 0)),
                safe_int(added_status.get("beaten", 0)),
                safe_int(added_status.get("toplay", 0)),
                safe_int(added_status.get("dropped", 0)),
                safe_int(added_status.get("playing", 0)),
                safe_int(esrb_rating.get("id")) if esrb_rating else None
            ))
        
        # Insertar en tabla games
        cursor.executemany("""
            INSERT INTO games (
                id, name, released, rating, ratings_count, metacritic, playtime,
                status_yet, status_owned, status_beaten, status_toplay, status_dropped, status_playing,
                esrb_rating_id
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO UPDATE SET
                name = EXCLUDED.name,
                released = EXCLUDED.released,
                rating = EXCLUDED.rating,
                ratings_count = EXCLUDED.ratings_count,
                metacritic = EXCLUDED.metacritic,
                playtime = EXCLUDED.playtime,
                status_yet = EXCLUDED.status_yet,
                status_owned = EXCLUDED.status_owned,
                status_beaten = EXCLUDED.status_beaten,
                status_toplay = EXCLUDED.status_toplay,
                status_dropped = EXCLUDED.status_dropped,
                status_playing = EXCLUDED.status_playing,
                esrb_rating_id = EXCLUDED.esrb_rating_id,
                updated = CURRENT_TIMESTAMP
        """, games_data)
        
        conn.commit()
        cursor.close()
        conn.close()
        
        print(f"Carga completada: {len(games_data)} juegos insertados")
        return {"statusCode": 200, "body": f"Cargados {len(games_data)} juegos"}
        
    except Exception as e:
        print(f"Error en Lambda 3: {e}")
        raise
'''

# Guardar handler 
with open("src/lambda_load_to_rds/handler.py", "w", encoding="utf-8") as f:
    f.write(lambda3_handler)

print("✅ Celda 10 ejecutada correctamente")
print("   - Handler para Lambda 3 creado: src/lambda_load_to_rds/handler.py")
print("   - Listo para empaquetar y desplegar en AWS Lambda")

✅ Celda 10 ejecutada correctamente
   - Handler para Lambda 3 creado: src/lambda_load_to_rds/handler.py
   - Listo para empaquetar y desplegar en AWS Lambda


In [9]:
# CELDA 11: CREAR HANDLER PARA LAMBDA 2 (EXTRACCIÓN DIARIA)
# Fuente: Adaptado de 20.AWS - LAMBDA.ipynb

lambda2_handler = '''

def lambda_handler(event, context):
    """Lambda 2: Extracción diaria de juegos actualizados"""
    try:
        # 1. Calcular fecha de hoy
        today = datetime.now().strftime("%Y-%m-%d")
        yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
        
        print(f"📅 Extrayendo juegos actualizados: {yesterday} → {today}")
        
        # 2. Parámetros para API de RAWG
        params = {
            "key": os.environ["RAWG_API_KEY"],
            "page_size": 100,
            "dates": f"{yesterday},{today}",
            "page": 1
        }
        
        all_games = []
        has_more = True
        
        # 3. Paginación para obtener todos los juegos del día
        while has_more:
            response = requests.get("https://api.rawg.io/api/games", params=params, timeout=10)
            response.raise_for_status()
            
            data = response.json()
            all_games.extend(data.get("results", []))
            
            # Verificar si hay más páginas
            if data.get("next"):
                params["page"] += 1
            else:
                has_more = False
        
        print(f"✅ Extraídos {len(all_games)} juegos actualizados")
        
        # 4. Subir a S3
        if all_games:
            s3_client = boto3.client("s3")
            filename = f"daily_{today.replace('-', '')}.json"
            s3_key = f"raw/{filename}"
            
            s3_client.put_object(
                Bucket=os.environ["S3_BUCKET_NAME"],
                Key=s3_key,
                Body=json.dumps(all_games, indent=2),
                ContentType="application/json"
            )
            
            print(f"📤 Subido a S3: s3://{os.environ['S3_BUCKET_NAME']}/{s3_key}")
            return {"statusCode": 200, "body": f"Extraídos {len(all_games)} juegos"}
        else:
            print("ℹ️  No hay juegos actualizados hoy")
            return {"statusCode": 200, "body": "No hay juegos nuevos"}
            
    except Exception as e:
        print(f"❌ Error en Lambda 2: {e}")
        raise
'''

# Guardar handler
with open("src/lambda_daily/handler.py", "w", encoding="utf-8") as f:
    f.write(lambda2_handler)


print("✅ Celda 11 ejecutada correctamente")
print("   - Handler para Lambda 2 creado: src/lambda_daily/handler.py")
print("   - Listo para configurar con EventBridge (ejecución diaria)")

✅ Celda 11 ejecutada correctamente
   - Handler para Lambda 2 creado: src/lambda_daily/handler.py
   - Listo para configurar con EventBridge (ejecución diaria)


In [14]:
# CELDA 12: INSTRUCCIONES PARA DESPLIEGUE EN AWS

# Crear carpeta docs si no existe
os.makedirs("docs", exist_ok=True)

aws_instructions = """
# INSTRUCCIONES PARA COMPLETAR LA FASE 01 EN AWS

## PASO 1: CREAR INSTANCIA RDS
1. Ir a AWS Console → RDS → Create database
2. Seleccionar: PostgreSQL 13+
3. DB instance identifier: rawg-games-db
4. Master username: postgres
5. Master password: [tu_contraseña_segura]
6. DB instance class: db.t3.micro (Free Tier)
7. Storage: 20 GB
8. VPC security group: Permitir conexiones desde Lambda
9. Database options: Initial database name = rawg_games_db

## PASO 2: CONFIGURAR BUCKET S3
1. Bucket ya existe: rawg-data-lake-manuel-39
2. En Permissions → Bucket Policy, añadir permisos para Lambda:
   {
     "Version": "2012-10-17",
     "Statement": [
       {
         "Effect": "Allow",
         "Principal": {"Service": "lambda.amazonaws.com"},
         "Action": ["s3:GetObject", "s3:PutObject"],
         "Resource": "arn:aws:s3:::rawg-data-lake-manuel-39/*"
       }
     ]
   }

## PASO 3: CREAR FUNCIONES LAMBDA
### Lambda 3 (Carga a RDS):
1. Crear función: lambda_load_to_rds
2. Runtime: Python 3.9
3. Architecture: x86_64
4. Upload from: .zip file (empaquetar src/lambda_load_to_rds/)
5. Variables de entorno:
   - DB_HOST: [endpoint-de-tu-RDS]
   - DB_NAME: rawg_games_db  
   - DB_USER: postgres
   - DB_PASSWORD: [tu_contraseña]
   - S3_BUCKET_NAME: rawg-data-lake-manuel-39

### Lambda 2 (Extracción diaria):
1. Crear función: lambda_daily
2. Mismo runtime y arquitectura
3. Variables de entorno:
   - RAWG_API_KEY: 53721c9ba8a74e818865f9bb81ccb779
   - S3_BUCKET_NAME: rawg-data-lake-manuel-39

## PASO 4: CONFIGURAR AUTOMATIZACIÓN
### Trigger S3 para Lambda 3:
1. Ir a S3 → rawg-data-lake-manuel-39 → Properties → Event notifications
2. Crear notificación:
   - Prefix: raw/
   - Events: ObjectCreated (All)
   - Destination: Lambda function → lambda_load_to_rds

### EventBridge para Lambda 2:
1. Ir a EventBridge → Rules → Create rule
2. Schedule pattern: Rate(1 day)
3. Target: Lambda function → lambda_daily

## PASO 5: EJECUTAR HISTÓRICA (una sola vez)
1. Subir archivo histórico a S3:
   - s3://rawg-data-lake-manuel-39/raw/historical_20260201.json
2. Esto activará automáticamente Lambda 3 para cargar datos en RDS

# FASE 01 COMPLETADA

"""

# Guardar instrucciones 
with open("docs/AWS_DEPLOYMENT_INSTRUCTIONS.md", "w", encoding="utf-8") as f:
    f.write(aws_instructions)

print("✅ Celda 12 ejecutada correctamente")
print("   - Carpeta docs/ creada")
print("   - Instrucciones de despliegue guardadas: docs/AWS_DEPLOYMENT_INSTRUCTIONS.md")
print("   - Guía completa para implementar en AWS")
print("\n🎯 RESUMEN FINAL:")
print("   ✅ Script SQL para RDS: 02_create_rawg_database_rds.sql")
print("   ✅ Estructura de Lambdas: src/lambda_*/")
print("   ✅ Handlers listos: Lambda 2 y Lambda 3")
print("   ✅ Instrucciones detalladas para AWS")
print("\n🎸 Guitarra Profesional bien Afinada - Fase 01 lista para AWS")

✅ Celda 12 ejecutada correctamente
   - Carpeta docs/ creada
   - Instrucciones de despliegue guardadas: docs/AWS_DEPLOYMENT_INSTRUCTIONS.md
   - Guía completa para implementar en AWS

🎯 RESUMEN FINAL:
   ✅ Script SQL para RDS: 02_create_rawg_database_rds.sql
   ✅ Estructura de Lambdas: src/lambda_*/
   ✅ Handlers listos: Lambda 2 y Lambda 3
   ✅ Instrucciones detalladas para AWS

🎸 Guitarra Profesional bien Afinada - Fase 01 lista para AWS
